# Vector Embeddings: From Intuition to Semantic Search

**Chapter 01 of [ai-musings](../README.md)**

By the end of this notebook you will be able to:

1. Explain what an embedding is and why "close vectors mean similar meaning".
2. Implement **dot product**, **cosine similarity** and **Euclidean distance** from scratch.
3. See why **one-hot** vectors can't capture meaning, and learn **dense word vectors** from raw text.
4. Embed sentences with a **pretrained transformer model**.
5. Build a small **semantic search** engine.
6. **Visualize** high-dimensional embeddings in 2-D and cluster them.

> Run cells top to bottom (`Shift + Enter`). Every section ends with a short
> *Try it* prompt. Change something and re-run, because that's where the learning happens.

In [ ]:
# Setup: all of these are in requirements.txt
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (7, 5)
plt.rcParams["axes.grid"] = True

---
## Part 1: The intuition

An **embedding** is a list of numbers that describes something. Before we let a
neural network pick the numbers, let's pick them **by hand** so the idea is
concrete.

We'll describe each item with 3 hand-chosen features:

| feature | meaning |
|---|---|
| `is_animal` | 1 = definitely an animal, 0 = definitely not |
| `size` | 0 = tiny, 1 = huge |
| `is_food` | 1 = commonly eaten, 0 = not |

Each item becomes a point in 3-D space, and that point is its *embedding*.

In [ ]:
items = {
    #            is_animal  size  is_food
    "cat":       [0.95,     0.20, 0.05],
    "dog":       [0.95,     0.35, 0.05],
    "lion":      [0.95,     0.60, 0.00],
    "elephant":  [0.95,     0.95, 0.00],
    "chicken":   [0.90,     0.15, 0.80],
    "apple":     [0.00,     0.10, 0.95],
    "banana":    [0.00,     0.10, 0.90],
    "pizza":     [0.00,     0.25, 1.00],
    "car":       [0.00,     0.60, 0.00],
    "bus":       [0.00,     0.90, 0.00],
}
names = list(items)
X = np.array([items[n] for n in names])
print("Embedding matrix shape:", X.shape, "-> (num_items, num_dimensions)")
print("cat =", X[names.index("cat")])

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3-D projection)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], s=60)
for name, (a, s, f) in zip(names, X):
    ax.text(a, s, f, f"  {name}", fontsize=9)
ax.set_xlabel("is_animal"); ax.set_ylabel("size"); ax.set_zlabel("is_food")
ax.set_title("Hand-made 3-D embeddings")
plt.show()

Notice the **clusters**: animals group together, fruit and food group together,
and vehicles group together. *Chicken* sits between animals and food, which is
fair.

That's the core promise of embeddings: **geometry encodes meaning.**

Real embedding models do the same thing with hundreds of dimensions. They
**learn** the features from data instead of having a human choose them, so no
single dimension has a tidy name like `size`.

> **Try it:** add `"shark"`, `"cookie"` and `"bicycle"` to `items` and re-run. Do they land where you expect?

---
## Part 2: Measuring similarity

"Close together" needs a number. There are three common measures:

| Measure | Formula | Range | Intuition |
|---|---|---|---|
| **Dot product** | $a \cdot b = \sum_i a_i b_i$ | $(-\infty, \infty)$ | Big when the vectors point the same way *and* are long |
| **Cosine similarity** | $\dfrac{a \cdot b}{\lVert a\rVert\,\lVert b\rVert}$ | $[-1, 1]$ | Only the **angle** counts; length is ignored |
| **Euclidean distance** | $\sqrt{\sum_i (a_i-b_i)^2}$ | $[0, \infty)$ | Straight-line distance. **Lower** means more similar |

Cosine similarity is the default for text embeddings, because a vector's
*direction* carries the meaning and its length mostly doesn't.

Let's implement all three with plain NumPy.

In [ ]:
def dot(a, b):
    return float(np.sum(a * b))

def norm(a):
    return float(np.sqrt(np.sum(a * a)))

def cosine_similarity(a, b):
    return dot(a, b) / (norm(a) * norm(b))

def euclidean_distance(a, b):
    return float(np.sqrt(np.sum((a - b) ** 2)))

v = lambda name: X[names.index(name)]

for pair in [("cat", "dog"), ("cat", "elephant"), ("cat", "apple"), ("apple", "banana"), ("car", "bus")]:
    a, b = v(pair[0]), v(pair[1])
    print(f"{pair[0]:>8} vs {pair[1]:<8}  dot={dot(a, b):5.2f}   cos={cosine_similarity(a, b):5.2f}   dist={euclidean_distance(a, b):5.2f}")

### Check our work against scikit-learn

Never trust a hand-rolled implementation until it matches a library.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as sk_cos

ours = np.array([[cosine_similarity(a, b) for b in X] for a in X])
theirs = sk_cos(X)
print("Max difference vs sklearn:", np.abs(ours - theirs).max())
assert np.allclose(ours, theirs)

### Why cosine ignores length

Scale a vector up and its **direction** doesn't change, so cosine similarity
stays the same while dot product and distance both shift. This matters because
a longer document shouldn't count as "more similar" just for being longer.

In [ ]:
cat = v("cat")
big_cat = cat * 10   # same direction, 10x longer

print("cos(cat, 10*cat)  =", round(cosine_similarity(cat, big_cat), 4))
print("dot(cat, 10*cat)  =", round(dot(cat, big_cat), 4), " vs dot(cat, cat) =", round(dot(cat, cat), 4))
print("dist(cat, 10*cat) =", round(euclidean_distance(cat, big_cat), 4))

**Pro tip:** if you **normalize** every vector to length 1 first, then
*dot product = cosine similarity*. Vector databases rely on this trick because a
dot product is cheaper to compute.

In [ ]:
unit = lambda a: a / norm(a)
a, b = v("apple"), v("pizza")
print("cosine(a, b)          =", round(cosine_similarity(a, b), 6))
print("dot(unit(a), unit(b)) =", round(dot(unit(a), unit(b)), 6))

In [ ]:
# The full similarity matrix as a heatmap
def plot_similarity(matrix, labels, title):
    fig, ax = plt.subplots(figsize=(0.55 * len(labels) + 2, 0.55 * len(labels) + 1.5))
    im = ax.imshow(matrix, cmap="viridis", vmin=min(0, matrix.min()), vmax=1)
    ax.set_xticks(range(len(labels)), labels, rotation=60, ha="right")
    ax.set_yticks(range(len(labels)), labels)
    ax.grid(False)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if matrix[i, j] < 0.6 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

plot_similarity(ours, names, "Cosine similarity: hand-made embeddings")

> **Try it:** write `most_similar(name, k=3)`, which returns the `k` nearest
> items to `name` by cosine similarity. *(You'll reuse this idea all the way to
> semantic search.)*